# Answers: tiers 5 to 10

This notebook walks through our solutions for tiers 5 to 10. The code lives in `tierNN/solution.py` (that is what the checker runs); the notebook imports those files, shows the intermediate results, and checks every decrypted line against `hashes.txt`.

| Tier | Problem | Quantum method | Width |
|---|---|---|---|
| 5 | 10-bit key of the toy cipher, 2 known bytes | Grover search, 25 rounds | 19 qubits |
| 6 | 14-bit key, 1 known byte, 64 keys pass | Grover search with M = 64, 12 rounds, then checksum filter | 16 qubits |
| 7 | 14-bit phase of a one-qubit gate | Iterative phase estimation, one bit per circuit | 2 qubits |
| 8 | RSA, N = 35 | Shor order finding, one recycled control qubit | 9 qubits |
| 9 | RSA, N = 77 | Same, larger multiplier | 10 qubits |
| 10 | Diffie-Hellman, p = 83 | Shor discrete log: two phase estimations on one register | 10 qubits |

**To run it:** use a Python environment with the challenge's `requirements.txt` installed. The notebook finds the challenge repo either above this folder (when the team folder sits in `submissions/<team>/`) or next to it in `../Quantum-Cryptography-Challenge`.

In [1]:
import importlib.util
import math
import os
import random
import subprocess
import sys
import tempfile
from collections import Counter
from fractions import Fraction
from pathlib import Path

sys.dont_write_bytecode = True  # keep __pycache__ out of the submission folder

HERE = Path.cwd()
REPO = next(
    folder
    for folder in [HERE, *HERE.parents, HERE.parent / "Quantum-Cryptography-Challenge"]
    if (folder / "common" / "check.py").exists()
)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "tools"))

import yaml
from qiskit import QuantumCircuit, transpile

import check_submission
from common import check, load_locked
from common.toy_cipher import ToyCipher

CONFIG = yaml.safe_load((REPO / "tools" / "scoring_config.yaml").read_text())


def load_solution(tier):
    # Import tierNN/solution.py from this folder.
    path = HERE / f"tier{tier:02d}" / "solution.py"
    spec = importlib.util.spec_from_file_location(f"solution_tier{tier:02d}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def scored_metrics(circuit):
    # Transpile exactly like tools/check_submission.py and count gates the same way.
    compiled = transpile(
        circuit,
        basis_gates=CONFIG["basis_gates"],
        coupling_map=CONFIG["coupling_map"],
        initial_layout=CONFIG["initial_layout"],
        optimization_level=CONFIG["optimization_level"],
        seed_transpiler=CONFIG["seed_transpiler"],
    )
    return check_submission.compute_metrics(compiled)


answers = {}
print("challenge repo:", REPO)

challenge repo: /home/avopti/Documents/QiskitFallFest2026/Quantum-Cryptography-Challenge


---
## Tier 5: The Small Safe

**What the file hides.** A 10-bit key of the toy cipher. We know the plaintext starts with `QC`.

**Which method fits.** We have a yes/no test on keys (does this key decrypt the first two bytes to `QC`?) and no other structure to exploit, with one good key among $2^{10}$. That is unstructured search: **Grover's algorithm**. It needs about $\frac{\pi}{4}\sqrt{1024} \approx 25$ oracle calls where a classical search needs about 512 on average.

**Why it works.** Start in the uniform superposition of all keys. The oracle flips the sign of the right key; the diffusion step reflects every amplitude about the average. Together they rotate the state by $2\theta$ towards the right key, where $\sin\theta = 1/\sqrt{1024}$. After $R$ rounds the success probability is $\sin^2((2R+1)\theta)$.

### Building the oracle

Decrypting byte $i$ is $P = \text{PERM}(\text{SBOX}(C \oplus \text{mask}(k, i)))$. PERM and SBOX do not involve the key and are bijections, so we can move them to the known side:

$$\text{byte } i \text{ decrypts to } P \iff \text{mask}(k,i) = C \oplus \text{SBOX}^{-1}(\text{PERM}^{-1}(P)).$$

The right-hand side is a known constant, and each mask bit is `k[a] ^ k[b] ^ (k[c] & k[d])`. So each known byte is 8 small equations on the key qubits, and the S-box never has to be built as gates. Two more simplifications:

1. **Shared AND terms.** Bytes 0 and 1 reuse six of the same AND terms. "A and B hold" is the same as "A and (A xor B) hold", and when A and B share their AND term, A xor B is a plain parity. Those parities turn out to repeat, leaving only two independent ones.
2. **In-place evaluation.** An equation `k[j] ^ ... = t` is written onto key qubit `j` itself when no later equation still needs `k[j]`.

The equations after simplification:

In [2]:
t5 = load_solution(5)
equations, key_bits = t5.known_text_equations()


def show_equation(linear, product, constant):
    terms = [f"k{j}" for j in sorted(linear)]
    if product:
        terms.append(f"(k{product[0]} & k{product[1]})")
    return " ^ ".join(terms) + f" = {constant}"


for equation in equations:
    print(show_equation(*equation))

k0 ^ k8 ^ (k1 & k3) = 0
k1 ^ k9 ^ (k2 & k4) = 1
k2 ^ k8 ^ (k3 & k5) = 0
k3 ^ k9 ^ (k4 & k6) = 0
k4 ^ k8 ^ (k5 & k7) = 1
(k6 & k8) = 0
(k7 & k9) = 0
k7 ^ k9 ^ (k0 & k8) = 1
(k1 & k9) = 0
(k0 & k2) = 1
k5 ^ k9 = 0
k6 ^ k8 = 1


The oracle is `compute` → flip the phase if every check qubit has its wanted value (one multi-controlled Z) → `compute` in reverse. Here is the compute block. Qubits 0 to 9 are the key, the rest are scratch.

The Toffolis are drawn as CCX. In the real circuit they are relative-phase Toffolis (`RCCX`, 3 CX instead of 6). That is exact here: the oracle is $W^{-1} Z W$ with $Z$ diagonal, so the extra phases that RCCX adds in $W$ cancel against $W^{-1}$.

In [3]:
gates, checks, n_scratch = t5.plan_oracle(equations, key_bits)
compute = QuantumCircuit(key_bits + n_scratch)
for gate in gates:
    compute.cx(gate[1], gate[2]) if gate[0] == "cx" else compute.ccx(gate[1], gate[2], gate[3])
print(f"{n_scratch} scratch qubits, {sum(g[0] == 'ccx' for g in gates)} Toffolis, {sum(g[0] == 'cx' for g in gates)} CX")
print("marked when:", ", ".join(f"q{qubit}={wanted}" for qubit, wanted in checks))
print(compute.draw("text", fold=-1))

7 scratch qubits, 10 Toffolis, 11 CX
marked when: q10=0, q11=0, q12=0, q13=1, q14=0, q15=1, q16=0, q1=1, q2=0, q3=0, q4=1, q7=1
                                                                                                               
 q_0: ────────────■───────────────────■─────────────────────────────────────────────────────────────────────■──
                  │                   │                      ┌───┐┌───┐                                     │  
 q_1: ────────────┼─────────■─────────┼───────────────────■──┤ X ├┤ X ├─────────────────────────────────────┼──
                  │         │         │                   │  └─┬─┘└─┬─┘┌───┐┌───┐                           │  
 q_2: ────────────■─────────┼─────────┼───────────────────┼────┼────■──┤ X ├┤ X ├───────────────────────────┼──
                  │         │         │                   │    │    │  └─┬─┘└─┬─┘┌───┐┌───┐                 │  
 q_3: ────────────┼─────────┼─────────┼───────────────────■────┼────┼────┼────■──┤ X ├┤ 

**Testing the oracle without touching the challenge key.** We encrypt `QC` under made-up keys, build the oracle for each made-up ciphertext, and compare it against the reference `ToyCipher` on all 1024 inputs. It must mark exactly the keys that really decrypt to `QC`.

In [4]:
rng = random.Random(2026)
for _ in range(25):
    made_up_key = rng.randrange(2 ** key_bits)
    made_up_ct = ToyCipher(made_up_key, key_bits).encrypt(b"QC")
    eqs = t5.simplify(
        t5.byte_equations(key_bits, 0, made_up_ct[0], ord("Q"))
        + t5.byte_equations(key_bits, 1, made_up_ct[1], ord("C"))
    )
    g, c, s = t5.plan_oracle(eqs, key_bits)
    marked = {k for k in range(2 ** key_bits) if t5.classical_oracle(k, g, c, key_bits + s)}
    truth = {k for k in range(2 ** key_bits) if ToyCipher(k, key_bits).decrypt(made_up_ct) == b"QC"}
    assert marked == truth and made_up_key in marked
print("oracle agrees with ToyCipher on 25 made-up keys (all 1024 inputs each)")

oracle agrees with ToyCipher on 25 made-up keys (all 1024 inputs each)


### The Grover circuit and its cost

In [5]:
circuit5, rounds5 = t5.grover_circuit(equations, key_bits)
theta = math.asin(1 / math.sqrt(2 ** key_bits))
print(f"qubits: {circuit5.num_qubits}  (10 key + {n_scratch} scratch + 2 ancillas for the multi-controlled gates)")
print(f"Grover rounds: {rounds5}, predicted success probability {math.sin((2 * rounds5 + 1) * theta) ** 2:.4f}")
print("scored metrics:", scored_metrics(transpile(circuit5, basis_gates=t5.BASIS, optimization_level=2, seed_transpiler=1)))

qubits: 19  (10 key + 7 scratch + 2 ancillas for the multi-controlled gates)
Grover rounds: 25, predicted success probability 0.9995
scored metrics: {'two_qubit_gates': 4750, 'depth': 4960, 'total_gates': 14588, 'active_qubits': 19}


### Run it

In [6]:
counts5 = t5.run_search()
shots5 = sum(counts5.values())
for key, n in counts5.most_common(5):
    print(f"key {key:4d} = {key:010b}   {n}/{shots5} shots")

key5 = counts5.most_common(1)[0][0]
ciphertext5 = bytes.fromhex(load_locked(5)["ciphertext_hex"])
answers[5] = ToyCipher(key5, key_bits).decrypt(ciphertext5).decode()
print()
print(answers[5])
check(5, answers[5]);

key  263 = 0100000111   256/256 shots

QC-05: count the lamps on the Long Walk: there are nineteen
PASS: tier 5 hash matches. Well done.


**Classical cost.** Trying keys one by one takes 512 decryptions on average, 1024 at worst. Grover uses 25 oracle calls, a $\sqrt{N}$ speed-up. Each call is far more expensive than a classical decryption, so at this size the classical search wins easily.

---
## Tier 6: The Bigger Safe

**What the file hides.** The same cipher with a 14-bit key. Only the first byte `Q` is trusted, plus a public checksum of the whole plaintext.

**Which method fits.** Still Grover, but one known byte is only 8 conditions on 14 key bits, so many keys pass. With $M$ marked keys out of $N$ the rotation angle is $\sin\theta = \sqrt{M/N}$ and the right number of rounds drops to $\frac{\pi}{4}\sqrt{N/M}$. Using the tier 5 count would overshoot badly.

**How many keys pass?** Look at the equations: equation $j$ is `k[j] ^ k[8 + j mod 6] ^ (k[j+1] & k[j+3]) = t[j]`. It only reads key bits with a larger index than $j$, so all 8 can be evaluated in place on qubits 0 to 7, in order. An in-place reversible circuit is a bijection, so for each of the $2^6$ values of $k_8..k_{13}$ exactly one value of $k_0..k_7$ matches:

$$M = 2^{14-8} = 64 \text{ exactly}, \qquad N/M = 256, \qquad R = \lfloor \tfrac{\pi}{4}\cdot 16 \rfloor = 12.$$

The oracle needs **no scratch qubits at all**: 14 key qubits plus 2 ancillas for the multi-controlled gates.

In [7]:
t6 = load_solution(6)
equations6, key_bits6 = t6.first_byte_equations()
for equation in equations6:
    print(show_equation(*equation))

gates6, checks6, n_scratch6 = t6.plan_oracle(equations6, key_bits6)
marked6 = t6.count_marked(equations6, key_bits6)
circuit6, rounds6 = t6.grover_circuit(equations6, key_bits6, n_marked=marked6)
theta6 = math.asin(math.sqrt(marked6 / 2 ** key_bits6))
print()
print(f"scratch qubits: {n_scratch6}, marked keys M = {marked6}")
print(f"qubits: {circuit6.num_qubits}, Grover rounds: {rounds6}")
print(f"predicted chance that a shot is a passing key: {math.sin((2 * rounds6 + 1) * theta6) ** 2:.4f}")
print("scored metrics:", scored_metrics(transpile(circuit6, basis_gates=t6.BASIS, optimization_level=2, seed_transpiler=1)))

k0 ^ k8 ^ (k1 & k3) = 0
k1 ^ k9 ^ (k2 & k4) = 0
k2 ^ k10 ^ (k3 & k5) = 1
k3 ^ k11 ^ (k4 & k6) = 1
k4 ^ k12 ^ (k5 & k7) = 0
k5 ^ k13 ^ (k6 & k8) = 0
k6 ^ k8 ^ (k7 & k9) = 1
k7 ^ k9 ^ (k8 & k10) = 1

scratch qubits: 0, marked keys M = 64
qubits: 16, Grover rounds: 12
predicted chance that a shot is a passing key: 0.9999
scored metrics: {'two_qubit_gates': 2064, 'depth': 2826, 'total_gates': 6948, 'active_qubits': 16}


**What if the guess for $M$ is off?** The success probability $\sin^2((2R+1)\theta)$ is flat near its peak, so a guess that is a little wrong costs little. A guess that is far off (for example assuming $M = 1$) rotates past the target and can do worse than random.

In [8]:
print("rounds  P(shot is a passing key)")
for rounds in (1, 4, 8, 10, 11, 12, 13, 14, 18, 25, 100):
    print(f"{rounds:6d}  {math.sin((2 * rounds + 1) * theta6) ** 2:.3f}")

rounds  P(shot is a passing key)
     1  0.035
     4  0.285
     8  0.764
    10  0.935
    11  0.983
    12  1.000
    13  0.986
    14  0.942
    18  0.542
    25  0.002
   100  0.000


### Run it, then filter with the checksum

In [9]:
counts6 = t6.run_search()
ciphertext6 = bytes.fromhex(load_locked(6)["ciphertext_hex"])
passing = {k: n for k, n in counts6.items() if ToyCipher(k, key_bits6).decrypt(ciphertext6[:1]) == b"Q"}
print(f"shots: {sum(counts6.values())}, distinct keys measured: {len(counts6)}")
print(f"shots that pass the first-byte test: {sum(passing.values())} ({len(passing)} distinct keys)")

survivors = t6.filter_candidates(counts6)
print(f"keys that also match the checksum {load_locked(6)['public_parameters']['checksum']}: {[k for k, _ in survivors]}")
answers[6] = survivors[0][1].decode()
print()
print(answers[6])
check(6, answers[6]);

shots: 4096, distinct keys measured: 65
shots that pass the first-byte test: 4095 (64 distinct keys)
keys that also match the checksum 221: [5401]

QC-06: the Thompson Library elevator stops twice on floor 8
PASS: tier 6 hash matches. Well done.


**Classical cost.** A classical search tests up to 16384 keys on the first byte and then runs the checksum on the 64 that pass. Grover finds a passing key in 12 oracle calls, but each shot returns a random one of the 64, so we need on the order of $64 \ln 64 \approx 270$ shots to be sure the right one has shown up (we take 4096). The checksum filter itself is classical in both cases.

---
## Tier 7: The Hidden Angle

**What the file hides.** A one-qubit gate $U$ with eigenvalue $e^{2\pi i \varphi}$, $\varphi = k/2^{14}$, and a circuit that prepares the matching eigenstate.

**Which method fits.** Reading an eigenvalue's phase is **quantum phase estimation**. A controlled-$U^{2^j}$ on the eigenstate leaves the eigenstate alone and kicks the phase $2\pi\varphi\,2^j$ back onto the control qubit.

**Our circuit: one bit per round.** Instead of 14 control qubits and an inverse QFT, we reuse one control qubit for 14 rounds (iterative, or "semiclassical", phase estimation). Round $m$ applies $U^{2^{13-m}}$, which puts the phase

$$2\pi\,\frac{k \bmod 2^{m+1}}{2^{m+1}} = \pi k_m + 2\pi\,\frac{k \bmod 2^{m}}{2^{m+1}}$$

on the control. The second term only involves bits measured in earlier rounds, so a phase gate cancels it. What remains is $\pi k_m$, and a Hadamard turns that into a certain 0 or 1. Because $\varphi$ has an exact 14-bit expansion, every round is deterministic.

$U^{2^j}$ is built from the matrix of $U$ by repeated squaring, which the tier rules allow. The matrix keeps $U$'s global phase, which matters once $U$ is controlled. We never compute an eigenvalue classically.

In [10]:
t7 = load_solution(7)
k7, log7 = t7.run_phase_estimation()
print("round  U power  counts        bit")
for row in log7:
    print(f"{row['bit']:5d}  {row['power']:7d}  {str(row['counts']):12s}  {row['value']}")
key_bits7 = format(k7, "014b")
print(f"\nk = {k7} = {key_bits7},  phi = {k7}/16384 = {k7 / 16384:.6f}")

round  U power  counts        bit
    0     8192  {'1': 64}     1
    1     4096  {'1': 64}     1
    2     2048  {'1': 64}     1
    3     1024  {'0': 64}     0
    4      512  {'0': 64}     0
    5      256  {'0': 64}     0
    6      128  {'1': 64}     1
    7       64  {'0': 64}     0
    8       32  {'0': 64}     0
    9       16  {'0': 64}     0
   10        8  {'1': 64}     1
   11        4  {'0': 64}     0
   12        2  {'1': 64}     1
   13        1  {'1': 64}     1

k = 13383 = 11010001000111,  phi = 13383/16384 = 0.816833


In [11]:
# The last round (bit 13), with the correction for the 13 bits found before it.
last_round = t7.round_circuit(13, k7 % 2 ** 13, 14)
print(last_round.draw("text"))
print("scored metrics (largest round):", scored_metrics(transpile(last_round, basis_gates=t7.BASIS, optimization_level=3)))

     ┌───┐                                  ┌────────────┐┌───┐┌─┐
q_0: ┤ H ├──────────────────────────────■───┤ P(-1.9907) ├┤ H ├┤M├
     ├───┤┌─────────────────────────┐┌──┴──┐└────────────┘└───┘└╥┘
q_1: ┤ X ├┤ U(1.1311,1.4721,1.4351) ├┤ U^1 ├────────────────────╫─
     └───┘└─────────────────────────┘└─────┘                    ║ 
c: 1/═══════════════════════════════════════════════════════════╩═
                                                                0 
scored metrics (largest round): {'two_qubit_gates': 2, 'depth': 11, 'total_gates': 17, 'active_qubits': 2}


In [12]:
answers[7] = t7.solve()
print(answers[7])
check(7, answers[7]);

QC-07: ask the night librarian for Folio 1870-Q, not 1870-R
PASS: tier 7 hash matches. Well done.


**Cost and caveats.**

- 14 circuits of 2 qubits and 2 CX each, instead of one 15-qubit circuit with about 200 CX. The checker scores the largest executed circuit, so this scores 2 CX on 2 qubits. It also raises the checker's "under 8 active qubits" review flag. `SCORING.md` names recycling a measured control qubit as a legitimate technique, but we must explain it in the write-up.
- The trade: total work is 14 circuit executions and the classical feed-forward between them.
- Classically, anyone holding the matrix of $U$ can diagonalise a 2x2 matrix instantly. The quantum advantage of phase estimation only appears when $U$ is given as a black box on many qubits.

---
## Tier 8: The Wrapped Key, N = 35

**What the file hides.** A key wrapped with textbook RSA, $c = k^e \bmod N$.

**Which method fits.** Unwrapping needs $d = e^{-1} \bmod (p-1)(q-1)$, so we must factor $N$. Factoring reduces to finding the order $r$ of some $a$ modulo $N$, and order finding is period finding: **Shor's algorithm**.

**Why it works.** $U|x\rangle = |a x \bmod N\rangle$ has eigenvalues $e^{2\pi i s/r}$, and $|1\rangle$ is an equal mix of its eigenstates. Phase estimation on $|1\rangle$ therefore returns $y/2^t \approx s/r$ for a random $s$. Continued fractions recover $r$, and if $r$ is even then $\gcd(a^{r/2} \pm 1, N)$ usually gives the factors.

**Our circuit.**

- **One control qubit, reused.** Same trick as tier 7: measure one phase bit, reset the control, feed the result forward as a phase correction. The work register is never measured in between, so all rounds see the same eigenstate. Here the feed-forward happens *inside* one circuit (`if_test`), because the work register must stay coherent across rounds.
- **$t = \lceil \log_2(N^2/2) \rceil$ rounds.** $r$ divides $\text{lcm}(p-1, q-1) < N/2$, so $2^t \ge 2r^2$, which is what continued fractions need.
- **Controlled multiplication** by $b = a^{2^j} \bmod N$ is a permutation of $1..N-1$. We write it as a product of swaps of two basis states. A swap is CX gates that make the two states differ in a single bit, one multi-controlled X on that bit, and the CX gates again. Controls are dropped when that only disturbs bit patterns the register can never hold ($0$ and values $\ge N$).
- **First round for free.** The register is exactly $|1\rangle$ there, so "multiply by $b$" is "write $b$": a few CX gates.

The base is $a = 2$, the smallest number coprime to $N$.

In [13]:
t8 = load_solution(8)
params8 = load_locked(8)["public_parameters"]
N8, a8 = params8["N"], 2
bits8 = t8.counting_bits(N8)
print(f"N = {N8}, a = {a8}, phase bits t = {bits8}")
print("multipliers a^(2^j) mod N, j = 0..t-1:", [pow(a8, 2 ** j, N8) for j in range(bits8)])

circuit8 = t8.order_finding_circuit(a8, N8)
print(f"qubits: {circuit8.num_qubits}  (1 control + {N8.bit_length()} work + {t8.N_ANCILLAS} ancillas)")
for b in sorted({pow(a8, 2 ** j, N8) for j in range(bits8 - 1)}):
    block = transpile(t8.controlled_multiplier(b, N8, N8.bit_length()), basis_gates=t8.BASIS, optimization_level=2, seed_transpiler=1)
    print(f"  controlled x{b:<2d} mod {N8}: {block.count_ops()['cx']} CX")
print("scored metrics:", scored_metrics(circuit8))

N = 35, a = 2, phase bits t = 10
multipliers a^(2^j) mod N, j = 0..t-1: [2, 4, 16, 11, 16, 11, 16, 11, 16, 11]
qubits: 9  (1 control + 6 work + 2 ancillas)
  controlled x2  mod 35: 722 CX
  controlled x4  mod 35: 663 CX
  controlled x11 mod 35: 530 CX
  controlled x16 mod 35: 519 CX
scored metrics: {'two_qubit_gates': 5053, 'depth': 10406, 'total_gates': 14920, 'active_qubits': 9}


In [14]:
def show_phase_table(counts, t, modulus):
    print("    y  shots  y/2^t    closest s/r")
    for y, n in sorted(counts.items()):
        fraction = Fraction(y, 2 ** t).limit_denominator(modulus // 2)
        print(f"{y:5d}  {n:5d}  {y / 2 ** t:.4f}   {fraction}")


counts8 = t8.run_order_finding(a8, N8)
show_phase_table(counts8, bits8, N8)

    y  shots  y/2^t    closest s/r
    0      3  0.0000   0
   85      1  0.0830   1/12
  170      1  0.1660   1/6
  171      3  0.1670   1/6
  256      3  0.2500   1/4
  341      1  0.3330   1/3
  425      1  0.4150   5/12
  427      1  0.4170   5/12
  512      3  0.5000   1/2
  597      3  0.5830   7/12
  683      1  0.6670   2/3
  853      2  0.8330   5/6
  939      1  0.9170   11/12


In [15]:
r8 = t8.order_from_counts(counts8, a8, N8)
half8 = pow(a8, r8 // 2, N8)
p8, q8 = math.gcd(half8 - 1, N8), math.gcd(half8 + 1, N8)
print(f"order r = {r8}   (check: {a8}^{r8} mod {N8} = {pow(a8, r8, N8)})")
print(f"a^(r/2) = {half8},  gcd({half8 - 1}, {N8}) = {p8},  gcd({half8 + 1}, {N8}) = {q8}")

d8 = pow(params8["e"], -1, (p8 - 1) * (q8 - 1))
key_bits8 = t8.unwrap_key(params8, p8, q8)
print(f"d = {d8},  k = {params8['wrapped_key']}^{d8} mod {N8} = {int(key_bits8, 2)} = {key_bits8}")

answers[8] = t8.solve()
print()
print(answers[8])
check(8, answers[8]);

order r = 12   (check: 2^12 mod 35 = 1)
a^(r/2) = 29,  gcd(28, 35) = 7,  gcd(30, 35) = 5
d = 5,  k = 23^5 mod 35 = 18 = 010010



QC-08: inside is a map of old tunnels under Ohio Stadium
PASS: tier 8 hash matches. Well done.


**Classical cost and an honest limit.** Trial division factors 35 in two steps. More to the point, our multiplier is built from the full table $x \mapsto bx \bmod N$, which takes $O(N)$ classical work and $O(N)$ gates. That is more than classical factoring costs, so this circuit shows the mechanism of Shor's algorithm but has no advantage. A scalable version replaces the permutation with modular arithmetic circuits ($O(n^2)$ to $O(n^3)$ gates for $n$-bit $N$).

---
## Tier 9: The Wrapped Key, N = 77

Identical method and code to tier 8. The work register grows to 7 qubits, $t$ grows to 12 rounds, and each multiplier is a permutation of 76 values instead of 34, so the circuit is roughly five times larger.

In [16]:
t9 = load_solution(9)
params9 = load_locked(9)["public_parameters"]
N9, a9 = params9["N"], 2
bits9 = t9.counting_bits(N9)
print(f"N = {N9}, a = {a9}, phase bits t = {bits9}")
print("multipliers a^(2^j) mod N, j = 0..t-1:", [pow(a9, 2 ** j, N9) for j in range(bits9)])

circuit9 = t9.order_finding_circuit(a9, N9)
print(f"qubits: {circuit9.num_qubits}  (1 control + {N9.bit_length()} work + {t9.N_ANCILLAS} ancillas)")
for b in sorted({pow(a9, 2 ** j, N9) for j in range(bits9 - 1)}):
    block = transpile(t9.controlled_multiplier(b, N9, N9.bit_length()), basis_gates=t9.BASIS, optimization_level=2, seed_transpiler=1)
    print(f"  controlled x{b:<2d} mod {N9}: {block.count_ops()['cx']} CX")
print("scored metrics:", scored_metrics(circuit9))

N = 77, a = 2, phase bits t = 12
multipliers a^(2^j) mod N, j = 0..t-1: [2, 4, 16, 25, 9, 4, 16, 25, 9, 4, 16, 25]


qubits: 10  (1 control + 7 work + 2 ancillas)
  controlled x2  mod 77: 2108 CX
  controlled x4  mod 77: 2208 CX
  controlled x9  mod 77: 2140 CX
  controlled x16 mod 77: 2194 CX


  controlled x25 mod 77: 2188 CX


scored metrics: {'two_qubit_gates': 23968, 'depth': 45696, 'total_gates': 70246, 'active_qubits': 10}


In [17]:
counts9 = t9.run_order_finding(a9, N9)
show_phase_table(counts9, bits9, N9)

    y  shots  y/2^t    closest s/r
    0      2  0.0000   0
  137      1  0.0334   1/30
  273      1  0.0667   1/15
  546      1  0.1333   2/15
  683      1  0.1667   1/6
  820      1  0.2002   1/5
 1092      1  0.2666   4/15
 1365      1  0.3333   1/3
 1368      1  0.3340   1/3
 1775      1  0.4333   13/30
 2185      1  0.5334   8/15
 2321      2  0.5667   17/30
 2459      1  0.6003   3/5
 2731      1  0.6667   2/3
 2867      1  0.7000   7/10
 3140      2  0.7666   23/30
 3277      3  0.8000   4/5
 3687      1  0.9001   9/10
 3960      1  0.9668   29/30


In [18]:
r9 = t9.order_from_counts(counts9, a9, N9)
half9 = pow(a9, r9 // 2, N9)
p9, q9 = math.gcd(half9 - 1, N9), math.gcd(half9 + 1, N9)
print(f"order r = {r9}   (check: {a9}^{r9} mod {N9} = {pow(a9, r9, N9)})")
print(f"a^(r/2) = {half9},  gcd({half9 - 1}, {N9}) = {p9},  gcd({half9 + 1}, {N9}) = {q9}")

d9 = pow(params9["e"], -1, (p9 - 1) * (q9 - 1))
key_bits9 = t9.unwrap_key(params9, p9, q9)
print(f"d = {d9},  k = {params9['wrapped_key']}^{d9} mod {N9} = {int(key_bits9, 2)} = {key_bits9}")

answers[9] = t9.solve()
print()
print(answers[9])
check(9, answers[9]);

order r = 30   (check: 2^30 mod 77 = 1)
a^(r/2) = 43,  gcd(42, 77) = 7,  gcd(44, 77) = 11
d = 43,  k = 59^43 mod 77 = 31 = 0011111



QC-09: follow it south past Gate 17 to a door painted scarlet
PASS: tier 9 hash matches. Well done.


---
## Tier 10: The Handshake

**What the file hides.** A Diffie-Hellman shared secret $B^a \bmod p$, where only $A = g^a \bmod p$ and $B$ are public. We need the exponent $a$: a discrete logarithm.

**Which method fits.** $f(x_1, x_2) = g^{x_1} A^{x_2} \bmod p$ is constant along the direction $(a, -1)$, a hidden period in two dimensions. That is **Shor's discrete-log algorithm**.

**Why it works.** On the subgroup generated by $g$ (order $q = 41$), let $U_g|x\rangle = |gx\rangle$ and $U_A|x\rangle = |Ax\rangle$. Since $U_A = U_g^a$ they share eigenvectors $|u_s\rangle$:

$$U_g|u_s\rangle = e^{2\pi i s/q}|u_s\rangle, \qquad U_A|u_s\rangle = e^{2\pi i s a/q}|u_s\rangle.$$

$|1\rangle$ is an equal mix of all $|u_s\rangle$. Phase estimation of $U_g$ gives a random $s$ and leaves the register (almost) in $|u_s\rangle$. Phase estimation of $U_A$ on the *same* register then gives $s a \bmod q$. So $a = (sa)\cdot s^{-1} \bmod q$.

**Our circuit.** Two 7-bit phase estimations in a row, both through the same recycled control qubit, on a 7-qubit work register. $2^7 = 128 > 2q$, so rounding $y\,q/2^7$ gives $s$ whenever the estimate is one of the two nearest values. The multipliers are built as in tiers 8 and 9, but only on the 41 subgroup elements. We find those with Euler's criterion ($x^{41} \equiv 1 \bmod 83$), which involves no discrete log.

In [19]:
t10 = load_solution(10)
params10 = load_locked(10)["public_parameters"]
p10, g10, A10, B10, q10 = (params10[name] for name in ("p", "g", "A", "B", "order"))
print(f"p = {p10}, g = {g10}, order q = {q10}, A = {A10}, B = {B10}")
print(f"subgroup size: {len(t10.subgroup(p10, q10))}")
print("multipliers g^(2^j):", [pow(g10, 2 ** j, p10) for j in range(t10.PHASE_BITS)])
print("multipliers A^(2^j):", [pow(A10, 2 ** j, p10) for j in range(t10.PHASE_BITS)])

circuit10 = t10.build_circuit()
print(f"qubits: {circuit10.num_qubits}  (1 control + {p10.bit_length()} work + {t10.N_ANCILLAS} ancillas)")
print("scored metrics:", scored_metrics(circuit10))

p = 83, g = 3, order q = 41, A = 11, B = 49
subgroup size: 41
multipliers g^(2^j): [3, 9, 81, 4, 16, 7, 49]
multipliers A^(2^j): [11, 38, 33, 10, 17, 40, 23]


qubits: 10  (1 control + 7 work + 2 ancillas)
scored metrics: {'two_qubit_gates': 13157, 'depth': 26112, 'total_gates': 37393, 'active_qubits': 10}


In [20]:
counts10 = t10.run_dlog(params10)
print("  y_g  y_A  shots    s   s*a   a = s*a / s mod 41")
for (y_g, y_a), n in sorted(counts10.items(), key=lambda item: -item[1])[:12]:
    s = round(y_g * q10 / 2 ** t10.PHASE_BITS) % q10
    s_a = round(y_a * q10 / 2 ** t10.PHASE_BITS) % q10
    exponent = (s_a * pow(s, -1, q10)) % q10 if s else "-"
    print(f"{y_g:5d} {y_a:4d} {n:6d} {s:4d} {s_a:5d}   {exponent}")
print("  ... (top 12 outcomes)")

  y_g  y_A  shots    s   s*a   a = s*a / s mod 41
   81  112      2   26    36   14
    3   44      2    1    14   14
  125   84      2   40    27   14
    0    0      2    0     0   -
   84   28      2   27     9   14
   25   94      2    8    30   14
   75   25      2   24     8   14
  112   37      1   36    12   14
  109  123      1   35    39   14
   50   57      1   16    18   37
   69   65      1   22    21   14
   94   31      1   30    10   14
  ... (top 12 outcomes)


In [21]:
votes10 = t10.exponent_votes(counts10, q10)
print("votes for a:", votes10.most_common(5), f"out of {sum(counts10.values())} shots")
a10 = votes10.most_common(1)[0][0]
print(f"a = {a10}   (check: {g10}^{a10} mod {p10} = {pow(g10, a10, p10)}, A = {A10})")
secret10 = pow(B10, a10, p10)
print(f"shared secret = {B10}^{a10} mod {p10} = {secret10} = {secret10:0{params10['key_width_bits']}b}")

answers[10] = t10.solve()
print()
print(answers[10])
check(10, answers[10]);

votes for a: [(14, 34), (37, 1), (39, 1), (19, 1), (34, 1)] out of 40 shots
a = 14   (check: 3^14 mod 83 = 11, A = 11)
shared secret = 49^14 mod 83 = 23 = 0010111



QC-10: knock four times, shout O-H, and wait for the I-O
PASS: tier 10 hash matches. Well done.


**Classical cost and an honest limit.** Trying exponents takes at most 41 multiplications; baby-step giant-step takes about $\sqrt{41} \approx 7$. As in tiers 8 and 9, our multipliers are compiled from a full table of the group, which costs more classical work than solving the problem directly. The circuit demonstrates the algorithm at toy size and is not a scalable implementation.

---
## All six lines, and the official checker

The first cell re-checks every line. The second runs `tools/check_submission.py` on this folder, exactly as the organisers will, and prints its report (written to a temporary folder, not into the submission).

In [22]:
for tier in sorted(answers):
    print(answers[tier])
print()
print("all correct:", all(check(tier, answers[tier]) for tier in sorted(answers)))

QC-05: count the lamps on the Long Walk: there are nineteen
QC-06: the Thompson Library elevator stops twice on floor 8
QC-07: ask the night librarian for Folio 1870-Q, not 1870-R
QC-08: inside is a map of old tunnels under Ohio Stadium
QC-09: follow it south past Gate 17 to a door painted scarlet
QC-10: knock four times, shout O-H, and wait for the I-O

PASS: tier 5 hash matches. Well done.
PASS: tier 6 hash matches. Well done.
PASS: tier 7 hash matches. Well done.
PASS: tier 8 hash matches. Well done.
PASS: tier 9 hash matches. Well done.
PASS: tier 10 hash matches. Well done.
all correct: True


In [23]:
with tempfile.TemporaryDirectory() as report_dir:
    result = subprocess.run(
        [sys.executable, str(REPO / "tools" / "check_submission.py"), str(HERE), "--out", report_dir],
        capture_output=True, text=True, cwd=REPO, env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"),
    )
print(result.stdout[result.stdout.index("| Tier"):] if "| Tier" in result.stdout else result.stdout + result.stderr)

| Tier | Status | Hash | Circuit run | 2Q gates | Depth | Gates | Active qubits | Metrics | Flags |
|---|---|---|---|---|---|---|---|---|---|
| 1 | not_attempted | - | no | - | - | - | - | no metrics | - |
| 2 | not_attempted | - | no | - | - | - | - | no metrics | - |
| 3 | not_attempted | - | no | - | - | - | - | no metrics | - |
| 4 | not_attempted | - | no | - | - | - | - | no metrics | - |
| 5 | full | match | yes | 4750 | 4960 | 14588 | 19 | ok | - |
| 6 | full | match | yes | 2064 | 2826 | 6948 | 16 | ok | - |
| 7 | full | match | yes | 2 | 11 | 16 | 2 | ok | scored circuit has under 8 active qubits |
| 8 | full | match | yes | 5053 | 10406 | 14920 | 9 | ok | - |
| 9 | full | match | yes | 23968 | 45696 | 70246 | 10 | ok | - |
| 10 | full | match | yes | 13157 | 26112 | 37393 | 10 | ok | - |


